# Hospital Daily Summary — Silver to Gold

Builds `careplus_db_ws.gold.hospital_daily_summary` with one row per hospital per calendar day, including test volumes, abnormal/critical rates, and day-over-day trend via LAG().

In [0]:
%sql
CREATE OR REPLACE TABLE careplus_db_ws.gold.hospital_daily_summary
COMMENT 'Hospital daily summary — one row per hospital per calendar day with test volumes and day-over-day trends'
AS
WITH daily_agg AS (
  SELECT
    lr.hospital_id,
    CAST(lr.test_date AS DATE)                                          AS summary_date,
    COUNT(*)                                                            AS total_tests,
    COUNT(DISTINCT lr.patient_id)                                       AS unique_patients_tested,
    COUNT(DISTINCT lr.doctor_id)                                        AS active_doctor_count,
    COUNT(CASE WHEN lr.result_status = 'Abnormal' THEN 1 END)          AS abnormal_test_count,
    COUNT(CASE WHEN lr.result_status = 'Critical' THEN 1 END)          AS critical_test_count
  FROM careplus_db_ws.silver.lab_results lr
  GROUP BY lr.hospital_id, CAST(lr.test_date AS DATE)
),
with_rates AS (
  SELECT
    da.*,
    CAST(da.abnormal_test_count * 100.0 / da.total_tests AS DECIMAL(5,2))  AS abnormal_rate_pct,
    CAST(da.critical_test_count * 100.0 / da.total_tests AS DECIMAL(5,2))  AS critical_rate_pct,
    LAG(da.total_tests) OVER (PARTITION BY da.hospital_id ORDER BY da.summary_date) AS prev_day_total_tests
  FROM daily_agg da
)
SELECT
  wr.hospital_id,
  h.hospital_name,
  wr.summary_date,
  c.day_name,
  c.is_weekend,
  wr.total_tests,
  wr.unique_patients_tested,
  wr.active_doctor_count,
  wr.abnormal_test_count,
  wr.critical_test_count,
  wr.abnormal_rate_pct,
  wr.critical_rate_pct,
  wr.prev_day_total_tests,
  CAST(
    CASE
      WHEN wr.prev_day_total_tests IS NULL OR wr.prev_day_total_tests = 0 THEN NULL
      ELSE (wr.total_tests - wr.prev_day_total_tests) * 100.0 / wr.prev_day_total_tests
    END AS DECIMAL(6,2)
  ) AS test_volume_change_pct
FROM with_rates wr
LEFT JOIN careplus_db_ws.silver.hospitals h
  ON wr.hospital_id = h.hospital_id
LEFT JOIN careplus_db_ws.silver.calender c
  ON c.date_key = CAST(date_format(wr.summary_date, 'yyyyMMdd') AS INT)

In [0]:
%sql
SELECT
  COUNT(*)                       AS total_rows,
  COUNT(DISTINCT hospital_id)    AS distinct_hospitals,
  COUNT(DISTINCT summary_date)   AS distinct_days,
  MIN(summary_date)              AS min_date,
  MAX(summary_date)              AS max_date,
  AVG(total_tests)               AS avg_daily_tests
FROM careplus_db_ws.gold.hospital_daily_summary

In [0]:
%sql
SELECT * FROM careplus_db_ws.gold.hospital_daily_summary
ORDER BY hospital_id, summary_date
LIMIT 10